# Suma de vectores en la GPU

**Laboratorio de aceleración de cómputo · Kaggle GPU**

En este ejercicio cada hilo CUDA suma un elemento de `A` y `B`, y almacena el resultado en `C`. La notebook conserva la bitácora de Kaggle; el código fuente mantenible también está disponible en `vector_add.cu`.

## Objetivos

- Verificar el entorno GPU y el compilador CUDA.
- Reservar memoria en host y device.
- Transferir datos con `cudaMemcpy`.
- Lanzar un hilo por elemento y validar el resultado completo.

> **Antes de ejecutar:** activa el acelerador **GPU** en la configuración de la notebook.

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Verificación del entorno

La GPU y la versión de `nvcc` quedan registradas antes de compilar el ejercicio.

## 2. Kernel y memoria CUDA

El índice global se calcula como `blockIdx.x * blockDim.x + threadIdx.x`. Cada hilo procesa un elemento y la condición `index < N` evita accesos fuera del arreglo. La celda siguiente escribe el programa CUDA en el entorno de Kaggle.

In [7]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define N (1 << 20) // 1,048,576 elementos

// 1. Definición del Kernel
__global__ void vectorAdd(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main(void) {
    size_t size = N * sizeof(float);

    // 2. Reserva de memoria en el Host (CPU)
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_c = (float*)malloc(size);

    // Inicialización de datos de entrada
    for (int i = 0; i < N; i++) {
        h_a[i] = sinf(i);
        h_b[i] = cosf(i);
    }

    // 3. Reserva de memoria en el Device (GPU)
    float *d_a, *d_b, *d_c;
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // 4. Copia de datos: Host -> Device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // 5. Configuración y lanzamiento del Kernel
    int blockSize = 256;
    int numBlocks = (N + blockSize - 1) / blockSize;
    vectorAdd<<<numBlocks, blockSize>>>(d_a, d_b, d_c, N);

    // 6. Copia de resultados: Device -> Host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // 7. Verificación de resultados en CPU
    printf("Verificación del primer elemento: c = %f (esperado: %f)\n",
           h_c[0], sinf(0) + cosf(0));
    printf("Verificación del último elemento: c[%d] = %f\n",
           N - 1, h_c[N - 1]);

    // 8. Liberación de memoria en ambos espacios
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    printf("Programa finalizado correctamente.\n");
    return 0;
}

Overwriting vector_add.cu


## 3. Compilación, ejecución y verificación

La salida de la siguiente celda confirma la compilación con `nvcc`, la suma del primer y último elemento y la finalización correcta del programa. Esta celda conserva la evidencia que ya estaba guardada en Kaggle.

In [8]:
!nvcc -O3 vector\_add.cu -o vector\_add 
!./vector\_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Verificación del primer elemento: c = 1.000000 (esperado: 1.000000)
Verificación del último elemento: c[1048575] = 0.172421
Programa finalizado correctamente.


## Resultado

La evidencia muestra que el kernel se compiló y ejecutó en GPU, que `c[0]` coincide con el valor esperado y que el último elemento también fue calculado. Para una validación exhaustiva, la versión del repositorio compara todos los elementos y reporta el número de errores.